In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import toad 
from sklearn.model_selection import train_test_split
from toad.plot import bin_plot


In [3]:
# 读取你导出的 CSV 文件
df = pd.read_csv('../03_clean_data/cleaned_data_202604.csv.csv')

#  确定特征和标签
# 删掉 ID 列（它对预测没用）和标签列本身
X = df.drop(columns=['SeriousDlqin2yrs', 'Id'])
y = df['SeriousDlqin2yrs']

# 划分数据集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3,       # 30% 做考试题（测试集），70% 做练习题（训练集）
    random_state=42,     # 随机种子：保证你下次运行结果跟我一样
    stratify=y           # 分层抽样：保证训练集和测试集里坏人的比例都是 6% 左右
)

print(f"训练集大小: {len(X_train)}, 坏人占比: {y_train.mean():.2%}")

训练集大小: 104990, 坏人占比: 6.68%


In [ ]:
#  为了方便，先把训练集的 X 和 y 合并
train_selected = pd.concat([X_train, y_train], axis=1)
# 特征筛选（toad.selection.select）
# 这一步会根据 IV 值、缺失率、相关性，自动帮你删掉没用的特征
# iv=0.02 表示 IV 低于 0.02 的特征会被剔除
X_selected, drop_lst = toad.selection.select(
    train_selected, 
    target='SeriousDlqin2yrs', 
    empty=0.9, iv=0.02, corr=0.7, 
    return_drop=True
)
print(f"剔除的特征清单: {drop_lst}")

#  分箱（toad.transform.Combiner）
combiner = toad.transform.Combiner()
# 这里是“学习”阶段，只看训练集
combiner.fit(X_selected, y='SeriousDlqin2yrs', method='chi') 

# 预览分箱结果（以 age 为例）
print(f"年龄的分箱切分点: {combiner.export()['age']}")

剔除的特征清单: {'empty': array([], dtype=float64), 'iv': array([], dtype=object), 'corr': array([], dtype=object)}
年龄的分箱切分点: [29, 37, 44, 45, 48, 54, 58, 63, 68]


In [15]:
#初始化WOE转换器
woe_transformer = toad.transform.WOETransformer()

#让woe_transformer学习利用分箱完成的结果计算woe1值
woe_transformer.fit(combiner.transform(X_selected),X_selected['SeriousDlqin2yrs'])

#真正计算woe值，并且把训练集和测试集都转换成woe值
train_woe = woe_transformer.transform(combiner.transform(X_selected))
test_woe = woe_transformer.transform(combiner.transform(X_test[X_selected.columns.drop('SeriousDlqin2yrs')]))

print("WOE 转化完成后的训练集数据预览：")
train_woe.head()


WOE 转化完成后的训练集数据预览：


,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30_59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60_89DaysPastDueNotWorse,NumberOfDependents,dr_level,SeriousDlqin2yrs
38099,-1.144343,0.255743,-0.502178,0.223741,0.272532,-0.047840,-0.370597,-0.177758,-0.271608,0.200568,0.014417,-8.856091
111963,0.289619,0.327995,0.876001,-0.178854,-0.449368,-0.146096,-0.370597,-0.254735,-0.271608,0.303225,0.014417,-8.856091
115701,-1.390681,0.005267,-0.502178,-0.147685,0.308803,-0.118386,-0.370597,-0.177758,-0.271608,0.105600,0.014417,-8.856091
119223,-1.634130,0.135157,-0.502178,-0.066867,-0.211729,0.747188,-0.370597,0.231772,-0.271608,-0.147298,0.524749,-8.856091
88050,-0.329089,-0.336218,-0.502178,-0.378240,-0.211729,0.037697,-0.370597,1.332300,-0.271608,-0.147298,-0.223822,-8.856091


In [18]:
from imblearn.over_sampling import SMOTE

# 1. 准备特征数据（这是经过 WOE 转换后的）
X_train_final = train_woe.drop(columns=['SeriousDlqin2yrs'])

# 2. 【关键修正】准备标签数据（必须用原始的 0/1 标签，不要用 WOE 化后的标签）
# 假设你原始的带有 0/1 标签的数据还在 X_selected 里
y_train_final = X_selected['SeriousDlqin2yrs'].astype('int') 

# 3. 执行 SMOTE
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_final, y_train_final)

print(f"平衡后，训练集样本数从 {len(y_train_final)} 增加到了 {len(y_train_res)}")
print(f"平衡后坏人比例: {y_train_res.mean():.2%}")

平衡后，训练集样本数从 104990 增加到了 195946
平衡后坏人比例: 50.00%


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 1. 训练模型
lr = LogisticRegression()
lr.fit(X_train_res, y_train_res)

# 2. 在“未见过”的测试集上预测（计算概率）
# [:, 1] 表示我们要的是“违约”的概率
y_pred_prob = lr.predict_proba(test_woe)[:, 1]

# 3. 计算 AUC 指标
auc_score = roc_auc_score(y_test, y_pred_prob)
print(f"模型在测试集上的 AUC 得分: {auc_score:.4f}")

模型在测试集上的 AUC 得分: 0.8592
